# JS08 — Tugas Lab 2: HDBSCAN pada Dataset Nyata

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Instruksi**
1. Pilih salah satu dataset nyata dari `sklearn.datasets` (iris atau digits).
2. Lakukan clustering dengan **HDBSCAN**.
3. Laporkan: jumlah cluster, banyaknya noise, dan visualisasi (pakai PCA/t-SNE untuk reduksi dimensi karena visualisasi maksimal 3D).
4. Analisis singkat: apakah hasil clustering HDBSCAN sesuai dengan label asli dataset?

**Prasyarat:** tidak perlu file eksternal — dataset diambil dari `sklearn.datasets`. `hdbscan` dipasang otomatis di cell pertama.

## Langkah 1 — Persiapan & Pemilihan Dataset

Ganti `DATASET = "digits"` menjadi `"iris"` bila ingin memakai dataset iris (4 fitur, 150 sampel). Digits punya 64 fitur (gambar 8×8), lebih menantang untuk HDBSCAN.

In [ ]:
# Instalasi hdbscan bila belum ada
try:
    import hdbscan  # noqa: F401
    print("hdbscan sudah terpasang")
except ImportError:
    %pip install -q hdbscan
    import hdbscan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             adjusted_rand_score, normalized_mutual_info_score)

DATASET = "digits"   # pilihan: "iris" atau "digits"

loader = {"iris": load_iris, "digits": load_digits}[DATASET]
bunch = loader()

X = bunch.data
y_true = bunch.target
fitur = list(bunch.feature_names)

print(f"Dataset        : {DATASET}")
print(f"Shape X        : {X.shape}")
print(f"Jumlah kelas   : {len(np.unique(y_true))} -> {[str(t) for t in bunch.target_names]}")

## Langkah 2 — Inspeksi Data

In [ ]:
df = pd.DataFrame(X, columns=fitur)
print("Jumlah sampel:", df.shape[0], "| jumlah fitur:", df.shape[1])
print("Sebaran label asli:", dict(zip(*np.unique(y_true, return_counts=True))))
df.iloc[:5, : min(8, df.shape[1])]

In [ ]:
df.describe().T.head(8)

## Langkah 3 — Clustering dengan HDBSCAN

Parameter dibiarkan *default* (`min_cluster_size=5`). HDBSCAN memberi label `-1` untuk titik yang dianggap noise.

In [ ]:
hdb = hdbscan.HDBSCAN().fit(X)
labels = hdb.labels_

n_cluster = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())

print(f"Jumlah cluster : {n_cluster}")
print(f"Jumlah noise   : {n_noise} ({n_noise / len(labels):.2%} dari {len(labels)} data)")
print()
print(pd.Series(labels).value_counts().sort_index()
        .rename_axis('label').to_frame('jumlah').T)

### [tambahan] Pengaruh `min_cluster_size`

Membantu memilih parameter: `min_cluster_size` kecil → cluster lebih banyak tapi noise menurun/naik tak menentu; terlalu besar → noise meningkat.

In [ ]:
def ringkas(X, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    n = len(set(labels[mask]))
    sil = dbi = np.nan
    if n >= 2 and np.bincount(labels[mask]).min() > 1:
        sil = silhouette_score(X[mask], labels[mask])
        dbi = davies_bouldin_score(X[mask], labels[mask])
    return n, int((~mask).sum()), sil, dbi

rows = []
for mcs in [5, 10, 15, 20, 30]:
    h = hdbscan.HDBSCAN(min_cluster_size=mcs).fit(X)
    n, noise, sil, dbi = ringkas(X, h.labels_)
    rows.append({'min_cluster_size': mcs, 'n_cluster': n, 'n_noise': noise,
                 'silhouette': sil, 'dbi': dbi})

tuning = pd.DataFrame(rows).set_index('min_cluster_size').round(4)
tuning

## Langkah 4 — Evaluasi Kualitas Cluster

Metrik internal dihitung **tanpa titik noise** (label `-1`), karena noise bukan anggota cluster. Untuk menjawab poin 4, dipakai juga **Adjusted Rand Index (ARI)** dan **Normalized Mutual Information (NMI)** — keduanya membandingkan hasil cluster dengan label asli (`1.0` = identik).

In [ ]:
mask = labels != -1

sil = silhouette_score(X[mask], labels[mask])
dbi = davies_bouldin_score(X[mask], labels[mask])
ari = adjusted_rand_score(y_true, labels)
nmi = normalized_mutual_info_score(y_true, labels)

hasil = pd.Series({
    'Silhouette Score': sil,
    'Davies-Bouldin Index': dbi,
    'Adjusted Rand Index': ari,
    'Normalized Mutual Info': nmi,
}).round(4)
hasil

**Cara membaca:** Silhouette tinggi & DBI rendah = cluster internal rapat dan terpisah. ARI/NMI tinggi = hasil HDBSCAN sejalan dengan label asli. Noise tetap dihitung pada ARI/NMI sebagai label tersendiri (`-1`).

## Langkah 5 — Visualisasi (PCA)

Data digits/iris berdimensi 4–64, jadi direduksi ke 2D dengan PCA (di-*scale* dulu agar tiap fitur setara). Panel kiri = hasil HDBSCAN, panel kanan = label asli sebagai pembanding.

In [ ]:
def scatter_embedding(ax, emb, labels, judul, noise_khusus=True):
    unik = sorted(set(labels))
    warna = plt.cm.tab10(np.linspace(0, 1, max(1, len([u for u in unik if u != -1]))))
    ci = 0
    for k in unik:
        m = labels == k
        if k == -1 and noise_khusus:
            ax.scatter(emb[m, 0], emb[m, 1], marker='x', c='k', s=25,
                       label=f'noise (n={m.sum()})')
        else:
            ax.scatter(emb[m, 0], emb[m, 1], s=28, alpha=0.85, color=warna[ci % len(warna)],
                       edgecolor='k', linewidth=0.3, label=f'cluster {k} (n={m.sum()})')
            ci += 1
    ax.set_title(judul)
    ax.set_xlabel('Komponen 1')
    ax.set_ylabel('Komponen 2')
    ax.legend(fontsize=8, loc='best')

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=0)
X_pca = pca.fit_transform(X_scaled)
print(f"Total explained variance PCA 2 komponen: {pca.explained_variance_ratio_.sum():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter_embedding(axes[0], X_pca, labels, f'PCA — HDBSCAN ({n_cluster} cluster, {n_noise} noise)')
scatter_embedding(axes[1], X_pca, y_true, f'PCA — label asli ({len(np.unique(y_true))} kelas)', noise_khusus=False)
plt.tight_layout()
plt.show()

### [tambahan] Visualisasi t-SNE

t-SNE sering memisahkan kelompok non-linier lebih baik daripada PCA, tetapi **hanya untuk visualisasi** — jarak antar kelompok tidak bisa dibaca secara kuantitatif. Jalankan bila PCA terlihat kurang memisahkan.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=0)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter_embedding(axes[0], X_tsne, labels, f't-SNE — HDBSCAN ({n_cluster} cluster, {n_noise} noise)')
scatter_embedding(axes[1], X_tsne, y_true, f't-SNE — label asli ({len(np.unique(y_true))} kelas)', noise_khusus=False)
plt.tight_layout()
plt.show()

## Langkah 6 — Perbandingan dengan Label Asli

Tabel silang di bawah menunjukkan label asli mana yang masuk ke cluster HDBSCAN mana. Dari sini terlihat apakah satu cluster didominasi satu kelas (sesuai) atau bercampur (tidak sesuai).

In [ ]:
ct = pd.crosstab(pd.Series(labels, name='cluster_HDBSCAN'),
                 pd.Series(y_true, name='label_asli'))
ct

In [ ]:
# [tambahan] "Purity" tiap cluster: proporsi kelas mayoritas di dalamnya
ct_cluster = ct.drop(index=-1, errors='ignore')  # noise bukan cluster
purity = (ct_cluster.max(axis=1) / ct_cluster.sum(axis=1)).round(3)
print("Purity per cluster (tanpa noise):")
print(purity.to_string())
print(f"\nRata-rata purity (tertimbang ukuran cluster): "
      f"{(ct_cluster.max(axis=1).sum() / ct_cluster.sum().sum()):.3f}")
print(f"ARI = {ari:.4f} | NMI = {nmi:.4f}")

## Langkah 7 — Analisis

**Apakah hasil clustering sesuai dengan label asli?** TBD

*(Isi dengan bukti: nilai ARI/NMI, cluster mana yang murni satu kelas, cluster mana yang bercampur, dan berapa persen data menjadi noise. Jelaskan juga mengapa HDBSCAN bisa berbeda dari label asli — misalnya kelas yang bertumpuk di ruang fitur, atau `min_cluster_size` yang terlalu besar/kecil.)*

---
## Kesimpulan Tugas Lab 2

**Dataset dipilih:** TBD

**Jumlah cluster / noise:** TBD

**Silhouette / DBI:** TBD

**ARI / NMI vs label asli:** TBD

**Kesesuaian dengan label asli & alasannya:** TBD